In [1]:
# !pip install shap
# %pip install -U google-genai

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.combine import SMOTEENN
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import shap
import os
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [3]:
# from google import genai

# client = genai.Client(
#     api_key=os.getenv("GEMINI_API_KEY")
# )

# print("Gemini client created successfully!")

# response = client.models.generate_content(
#     model="gemini-3.1-flash-lite",
#     contents="Explain customer churn prediction in simple terms."
# )

In [4]:
df = pd.read_excel("data_cleaned.xlsx")

In [5]:
df.shape

(7032, 33)

In [6]:
df.columns.to_list()

['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [7]:
location_cols = [
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude',
    'CLTV'
]

df.drop(columns=location_cols, inplace=True)

In [8]:
df.columns.to_list()

['CustomerID',
 'Count',
 'Country',
 'State',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'Churn Reason']

In [9]:
# Normalize categorical columns
# ---------------------------------------------
categorical_columns = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method"
]

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

In [10]:
df.columns.to_list()

['CustomerID',
 'Count',
 'Country',
 'State',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'Churn Reason']

In [11]:
service_cols = [
    'Phone Service',
    'Multiple Lines',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies'
]

df['Total Services'] = (df[service_cols] == 'yes').sum(axis=1)
df.head()

,CustomerID,Count,Country,State,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,Churn Reason,Total Services
0,3668-Qpybk,1,United States,California,male,no,no,no,2,yes,...,month-to-month,yes,mailed check,53.85,108.15,Yes,1,86,Competitor Made Better Offer,3
1,9237-Hqitu,1,United States,California,female,no,no,yes,2,yes,...,month-to-month,yes,electronic check,70.70,151.65,Yes,1,67,Moved,1
2,9305-Cdskc,1,United States,California,female,no,no,yes,8,yes,...,month-to-month,yes,electronic check,99.65,820.50,Yes,1,86,Moved,5
3,7892-Pookp,1,United States,California,female,no,yes,yes,28,yes,...,month-to-month,yes,electronic check,104.80,3046.05,Yes,1,84,Moved,6
4,0280-Xjgex,1,United States,California,male,no,no,yes,49,yes,...,month-to-month,yes,bank transfer (automatic),103.70,5036.30,Yes,1,89,Competitor Had Better Devices,6


In [12]:
df['New Customer'] = (df['Tenure Months'] <= 12).astype(int)

In [13]:
df['Avg Monthly Spend'] = (
    df['Total Charges'] /
    df['Tenure Months'].replace(0, 1)
)

In [14]:
df['Services_per_Month'] = (df['Total Services'] / df['Tenure Months'].replace(0,1))

In [15]:
df["High_Value"] = (
    df["Monthly Charges"] >
    df["Monthly Charges"].median()
)
print(df["Monthly Charges"].median())
df['High_Value'] = df['High_Value'].astype(int)
df['High_Value'].head()

70.35


0    0
1    1
2    1
3    1
4    1
Name: High_Value, dtype: int64

In [16]:
security_cols = [
    "Online Security",
    "Device Protection",
    "Tech Support"
]

df["Security Bundle"] = (
    (df["Online Security"] == "yes").astype(int) +
    (df["Device Protection"] == "yes").astype(int) +
    (df["Tech Support"] == "yes").astype(int)
)

In [17]:
df["Entertainment Bundle"] = (
    (df["Streaming TV"] == "yes").astype(int) +
    (df["Streaming Movies"] == "yes").astype(int)
)

In [18]:
df["Long-Term Customer"] = (
    df["Tenure Months"] > 24
).astype(int)

In [19]:
cols_to_drop = [
    'CustomerID',
    'Count',
    'Country',
    'State',
    'Churn Value',
    'Churn Reason',
    'Churn Score'
]

df.drop(columns=cols_to_drop, inplace=True)

In [20]:
df[
    [
        "Security Bundle",
        "Entertainment Bundle",
        "Long-Term Customer"
    ]
].head()

,Security Bundle,Entertainment Bundle,Long-Term Customer
0,1,0,0
1,0,0,0
2,1,2,0
3,2,2,1
4,1,2,1


In [21]:
df.columns.to_list()

['Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Total Services',
 'New Customer',
 'Avg Monthly Spend',
 'Services_per_Month',
 'High_Value',
 'Security Bundle',
 'Entertainment Bundle',
 'Long-Term Customer']

In [22]:
# Features and Target
X = df.drop(columns='Churn Label')
y = df['Churn Label']

In [23]:
print("Features:", X.shape)
print("Target:", y.shape)

Features: (7032, 27)
Target: (7032,)


In [24]:
y.value_counts()

Churn Label
No     5163
Yes    1869
Name: count, dtype: int64

In [25]:
y = y.map({
    'No': 0,
    'Yes': 1
})

In [26]:
df.shape

(7032, 28)

In [27]:
output_file = "data_cleaned_featued.xlsx"
df.to_excel(output_file, index=False)

print("Cleaned dataset saved as:", output_file)

Cleaned dataset saved as: data_cleaned_featued.xlsx


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [29]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nTraining Churn Distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting Churn Distribution:")
print(y_test.value_counts(normalize=True))

X_train: (5625, 27)
X_test: (1407, 27)

Training Churn Distribution:
Churn Label
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Testing Churn Distribution:
Churn Label
0    0.734186
1    0.265814
Name: proportion, dtype: float64


In [30]:
X.dtypes

Gender                      str
Senior Citizen              str
Partner                     str
Dependents                  str
Tenure Months             int64
Phone Service               str
Multiple Lines              str
Internet Service            str
Online Security             str
Online Backup               str
Device Protection           str
Tech Support                str
Streaming TV                str
Streaming Movies            str
Contract                    str
Paperless Billing           str
Payment Method              str
Monthly Charges         float64
Total Charges           float64
Total Services            int64
New Customer              int64
Avg Monthly Spend       float64
Services_per_Month      float64
High_Value                int64
Security Bundle           int64
Entertainment Bundle      int64
Long-Term Customer        int64
dtype: object

In [31]:
numeric = ['int64', 'float64']
numeric_features = X.select_dtypes(include = numeric).columns

categorical_features = X.select_dtypes(include = ['object',  'category']).columns

C:\Users\HP\AppData\Local\Temp\ipykernel_7540\2480478723.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include = ['object',  'category']).columns


In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [33]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [34]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(5625, 54)
(1407, 54)


In [35]:
data = pd.DataFrame(X_train_processed)
data.head()

,0,1,2,3,4,5,6,7,8,9,...,44,45,46,47,48,49,50,51,52,53
0,1.321816,0.981556,1.659900,1.269948,-0.664784,0.943761,-0.411937,0.991327,2.026707,-0.912483,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-0.267410,-0.971546,-0.562252,-0.671868,-0.664784,-0.849298,-0.440314,-1.008749,1.048902,-0.912483,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1.444064,0.837066,1.756104,0.784494,-0.664784,0.913760,-0.446574,0.991327,1.048902,-0.912483,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-1.204646,0.641092,-0.908326,-0.186414,1.504248,0.441857,1.262298,0.991327,-0.906708,0.258568,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.669826,-0.808787,-0.101561,-0.671868,-0.664784,-0.752245,-0.506913,-1.008749,0.071097,0.258568,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


# **MODEL TRAINING**

# ***Logistic Regression***

In [36]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [37]:
log_model.fit(X_train_processed, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [38]:
y_pred = log_model.predict(X_test_processed)
y_prob = log_model.predict_proba(X_test_processed)[:, 1]

In [39]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Results")
print("---------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("ROC-AUC  :", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression Results
---------------------
Accuracy : 0.7967
Precision: 0.6279
Recall   : 0.5775
F1 Score : 0.6017
ROC-AUC  : 0.8447

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1033
           1       0.63      0.58      0.60       374

    accuracy                           0.80      1407
   macro avg       0.74      0.73      0.73      1407
weighted avg       0.79      0.80      0.79      1407

ROC-AUC: 0.844711421486662


*The Logistic Regression baseline achieved an accuracy of 79.96% and a ROC-AUC of 0.842, demonstrating good overall discriminatory performance. However, the recall for the churn class was 56.68%, indicating that the model missed a considerable proportion of actual churners. Therefore, additional models and optimization techniques will be evaluated to improve churn detection.*

# **FINE TUNING LOGISTIC REGRESSION MODEL**

In [40]:
param_grid = [
    {
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    },
    {
        'penalty': ['l2'],
        'solver': ['lbfgs', 'newton-cg', 'sag'],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        'max_iter': [100, 250, 500, 1000]
    },
    {
        'penalty': ['elasticnet'],
        'solver': ['saga'],
        'C': [0.001, 0.01, 0.1, 1.0, 10.0],
        'l1_ratio': [0.1, 0.5, 0.9],
        'max_iter': [100, 250, 500, 1000]
    }
]

In [41]:
grid_search = GridSearchCV(
    estimator=log_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score = False
)

grid_search.fit(X_train_processed, y_train)
# grid_search.cv_results_

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'C': [0.001, 0.01, ...], 'penalty': ['l1', 'l2'], 'solver': ['liblinear', 'saga']}, {'C': [0.001, 0.01, ...], 'max_iter': [100, 250, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg', ...]}, ...]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the 

In [42]:
grid_df = pd.DataFrame(grid_search.cv_results_)
# grid_df.head()

In [43]:
grid_df[
    [
        "param_C",
        "param_penalty",
        "param_solver",
        "param_max_iter",
        "param_l1_ratio",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score").head()

,param_C,param_penalty,param_solver,param_max_iter,param_l1_ratio,mean_test_score,std_test_score,rank_test_score
16,10.0,l1,liblinear,NaN,NaN,0.621627,0.021064,1
149,10.0,elasticnet,saga,250.0,0.5,0.621191,0.021284,2
145,10.0,elasticnet,saga,250.0,0.1,0.621191,0.021284,2
154,10.0,elasticnet,saga,500.0,0.9,0.620968,0.021275,4
74,10.0,l2,sag,100.0,NaN,0.620915,0.021057,5


In [44]:
grid_search.best_params_

{'C': 10.0, 'penalty': 'l1', 'solver': 'liblinear'}

*BEST PARAMETERS FOR LOGISTIC REGRESSION*

{'C': 10.0, 'penalty': 'l1', 'solver': 'liblinear'}

In [45]:
best_log_model = grid_search.best_estimator_

In [46]:
y_pred_tuned = best_log_model.predict(X_test_processed)
y_prob_tuned = best_log_model.predict_proba(X_test_processed)[:, 1]
y_prob_tuned

array([0.03052482, 0.61381909, 0.00703388, ..., 0.13057468, 0.02522335,
       0.00155535], shape=(1407,))

In [47]:
accuracy_flog = accuracy_score(y_test, y_pred_tuned)
precision_flog = precision_score(y_test, y_pred_tuned)
recall_flog = recall_score(y_test, y_pred_tuned)
f1_flog = f1_score(y_test, y_pred_tuned)
roc_auc_flong = roc_auc_score(y_test, y_prob_tuned)

print("Tuned Logistic Regression Results")
print("---------------------")
print("Accuracy :", round(accuracy_flog, 4))
print("Precision:", round(precision_flog, 4))
print("Recall   :", round(recall_flog, 4))
print("F1 Score :", round(f1_flog, 4))
print("ROC-AUC  :", round(roc_auc_flong, 4))

print(classification_report(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

Tuned Logistic Regression Results
---------------------
Accuracy : 0.7946
Precision: 0.6246
Recall   : 0.5695
F1 Score : 0.5958
ROC-AUC  : 0.8443
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1033
           1       0.62      0.57      0.60       374

    accuracy                           0.79      1407
   macro avg       0.74      0.72      0.73      1407
weighted avg       0.79      0.79      0.79      1407

ROC-AUC: 0.8443076341686898


## **THRESHOLDED**

In [48]:
thresholds = np.arange(0.2, 0.81, 0.01)

best_f1 = 0
best_threshold = 0

for t in thresholds:
    preds = (y_prob_tuned >= t).astype(int)
    score = f1_score(y_test, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

final_preds = (y_prob_tuned >= best_threshold).astype(int)
print(best_threshold)
print("Tuned and Thresholded Logistic Regression Results")
print("---------------------")
print("Accuracy :", accuracy_score(y_test, final_preds))
print("Precision:", precision_score(y_test, final_preds))
print("Recall   :", recall_score(y_test, final_preds))
print("F1 Score :", f1_score(y_test, final_preds))
print(classification_report(y_test, final_preds))
# print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

0.45000000000000023
Tuned and Thresholded Logistic Regression Results
---------------------
Accuracy : 0.8002842928216063
Precision: 0.6177215189873417
Recall   : 0.6524064171122995
F1 Score : 0.6345903771131339
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      1033
           1       0.62      0.65      0.63       374

    accuracy                           0.80      1407
   macro avg       0.74      0.75      0.75      1407
weighted avg       0.80      0.80      0.80      1407



# **SMOTE**

In [49]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=100)

X_train_resampled, y_train_resampled = sm.fit_resample(
    X_train_processed,
    y_train
)

log_smote = LogisticRegression()
log_smote.fit(X_train_resampled, y_train_resampled)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [50]:
y_pred = best_log_model.predict(X_test_processed)
y_prob = best_log_model.predict_proba(X_test_processed)[:, 1]

In [51]:
accuracy_flog = accuracy_score(y_test, y_pred)
precision_flog = precision_score(y_test, y_pred)
recall_flog = recall_score(y_test, y_pred)
f1_flog = f1_score(y_test, y_pred)
roc_auc_flong = roc_auc_score(y_test, y_prob)

print("SMOTE Logistic Regression Results")
print("---------------------")
print("Accuracy :", round(accuracy_flog, 4))
print("Precision:", round(precision_flog, 4))
print("Recall   :", round(recall_flog, 4))
print("F1 Score :", round(f1_flog, 4))
print("ROC-AUC  :", round(roc_auc_flong, 4))

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

SMOTE Logistic Regression Results
---------------------
Accuracy : 0.7946
Precision: 0.6246
Recall   : 0.5695
F1 Score : 0.5958
ROC-AUC  : 0.8443
              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1033
           1       0.62      0.57      0.60       374

    accuracy                           0.79      1407
   macro avg       0.74      0.72      0.73      1407
weighted avg       0.79      0.79      0.79      1407

ROC-AUC: 0.8443076341686898


## ***Decision Tree Classifier***

In [52]:
dt_model = DecisionTreeClassifier(
    random_state=100,
    max_depth=6, 
    min_samples_leaf=8
)

dt_model.fit(X_train_processed, y_train)

y_pred_dt = dt_model.predict(X_test_processed)
y_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

In [53]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

print("Decision Tree Results")
print("---------------------")
print("Accuracy :", round(accuracy_dt, 4))
print("Precision:", round(precision_dt, 4))
print("Recall   :", round(recall_dt, 4))
print("F1 Score :", round(f1_dt, 4))
print("ROC-AUC  :", round(roc_auc_dt, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

Decision Tree Results
---------------------
Accuracy : 0.7875
Precision: 0.5984
Recall   : 0.6096
F1 Score : 0.604
ROC-AUC  : 0.8348

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.85      0.85      1033
           1       0.60      0.61      0.60       374

    accuracy                           0.79      1407
   macro avg       0.73      0.73      0.73      1407
weighted avg       0.79      0.79      0.79      1407



# **FINE TUNING DECISION TREE CLASSIFIER**

In [54]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'ccp_alpha': [0.0, 0.001]
}

In [55]:
grid_search = GridSearchCV(
    estimator=dt_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score = False
)

grid_search.fit(X_train_processed, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...dom_state=100)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'ccp_alpha': [0.0, 0.001], 'criterion': ['gini', 'entropy'], 'max_depth': [None, 5, ...], 'max_features': [None, 'sqrt', ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and 

In [56]:
grid_dt = pd.DataFrame(grid_search.cv_results_)
# grid_dt.sort_values("rank_test_score").head(5)

In [57]:
grid_search.best_params_

{'ccp_alpha': 0.001,
 'criterion': 'gini',
 'max_depth': 5,
 'max_features': None,
 'min_samples_leaf': 1,
 'min_samples_split': 2}

*BEST PARAMETERS FOR DECISION TREE*

{'ccp_alpha': 0.001,
 'criterion': 'gini',
 'max_depth': 5,
 'max_features': None,
 'min_samples_leaf': 1,
 'min_samples_split': 2}

In [58]:
best_dt_model = grid_search.best_estimator_

In [59]:
y_pred_tuned = best_dt_model.predict(X_test_processed)

y_prob_tuned = best_dt_model.predict_proba(X_test_processed)[:, 1]

In [60]:
accuracy_rf_tuned = accuracy_score(y_test, y_pred_tuned)
precision_rf_tuned = precision_score(y_test, y_pred_tuned)
recall_rf_tuned = recall_score(y_test, y_pred_tuned)
f1_rf_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_rf_tuned = roc_auc_score(y_test, y_prob_tuned)

print("Decision Tree Results")
print("---------------------")
print("Accuracy :", round(accuracy_rf_tuned, 4))
print("Precision:", round(precision_rf_tuned, 4))
print("Recall   :", round(recall_rf_tuned, 4))
print("F1 Score :", round(f1_rf_tuned, 4))
print("ROC-AUC  :", round(roc_auc_rf_tuned, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

Decision Tree Results
---------------------
Accuracy : 0.7868
Precision: 0.6
Recall   : 0.5936
F1 Score : 0.5968
ROC-AUC  : 0.822

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.86      0.86      1033
           1       0.60      0.59      0.60       374

    accuracy                           0.79      1407
   macro avg       0.73      0.73      0.73      1407
weighted avg       0.79      0.79      0.79      1407



# **SMOTEEN IT FURTHER**

In [61]:
sm = SMOTEENN()

X_train_resampled1, y_train_resampled1 = sm.fit_resample(
    X_train_processed,
    y_train
)

xr_train1,xr_test1,yr_train1,yr_test1=train_test_split(X_train_resampled1, y_train_resampled1,test_size=0.2)

model_dt_smote=DecisionTreeClassifier(criterion = "gini",random_state = 100,max_depth=6, min_samples_leaf=8)

# 4. Train the model
model_dt_smote.fit(X_train_resampled1, y_train_resampled1)

# 5. Evaluate on the untouched test set
y_predict = model_dt_smote.predict(X_test_processed)
y_probab = model_dt_smote.predict_proba(X_test_processed)[:, 1]

In [62]:
print("Decision Tree Results (SMOTEENN)")
print("---------------------")
print("Accuracy :", accuracy_score(y_test, y_predict))
print("Precision:", precision_score(y_test, y_predict))
print("Recall   :", recall_score(y_test, y_predict))
print("F1 Score :", f1_score(y_test, y_predict))
print("ROC-AUC  :", roc_auc_score(y_test, y_probab))

print("\nClassification Report:")
print(classification_report(y_test, y_predict))

Decision Tree Results (SMOTEENN)
---------------------
Accuracy : 0.6979388770433547
Precision: 0.4622222222222222
Recall   : 0.8342245989304813
F1 Score : 0.5948522402287894
ROC-AUC  : 0.7973104140890713

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.65      0.76      1033
           1       0.46      0.83      0.59       374

    accuracy                           0.70      1407
   macro avg       0.69      0.74      0.68      1407
weighted avg       0.79      0.70      0.72      1407



## ***Random Forest***

In [63]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    random_state = 100,
    max_depth=6, 
    min_samples_leaf=8
)

rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)
y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

In [64]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(accuracy_rf, 4))
print("Precision:", round(precision_rf, 4))
print("Recall   :", round(recall_rf, 4))
print("F1 Score :", round(f1_rf, 4))
print("ROC-AUC  :", round(roc_auc_rf, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Results
---------------------
Accuracy : 0.7918
Precision: 0.6494
Recall   : 0.4706
F1 Score : 0.5457
ROC-AUC  : 0.8452

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.86      1033
           1       0.65      0.47      0.55       374

    accuracy                           0.79      1407
   macro avg       0.74      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407



*Among the three evaluated models, Logistic Regression achieved the best overall performance, with an accuracy of 79.96% and ROC-AUC of 0.842. It also achieved the highest recall (56.68%) and F1-score (60.06%) for the churn class, indicating a stronger ability to identify customers at risk of churning. Decision Tree and Random Forest models showed comparatively lower performance across the evaluated metrics. Therefore, Logistic Regression was selected as the initial best-performing model.*

# **FINE TUNING RANDOM FOREST MODEL**

In [65]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt'],
    'bootstrap': [True]
}

In [66]:
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    return_train_score = False
)

grid_search.fit(X_train_processed, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...dom_state=100)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True], 'max_depth': [10, 20, ...], 'max_features': ['sqrt'], 'min_samples_leaf': [1, 2], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is d

In [67]:
grid_rf = pd.DataFrame(grid_search.cv_results_)
# grid_rf.sort_values("rank_test_score").head(10)

In [68]:
grid_search.best_params_

{'bootstrap': True,
 'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'n_estimators': 100}

*BEST PARAMETERS FOR RANDOM FOREST*

{'bootstrap': True,
 'max_depth': 10,
 'max_features': 'sqrt',
 'min_samples_leaf': 2,
 'min_samples_split': 5,
 'n_estimators': 100}

In [69]:
best_rf_model = grid_search.best_estimator_

In [70]:
y_pred_tuned = best_rf_model.predict(X_test_processed)

y_prob_tuned = best_rf_model.predict_proba(X_test_processed)[:, 1]

In [71]:
accuracy_rf_tuned = accuracy_score(y_test, y_pred_tuned)
precision_rf_tuned = precision_score(y_test, y_pred_tuned)
recall_rf_tuned = recall_score(y_test, y_pred_tuned)
f1_rf_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_rf_tuned = roc_auc_score(y_test, y_prob_tuned)

print("Random Forest Results")
print("---------------------")
print("Accuracy :", round(accuracy_rf_tuned, 4))
print("Precision:", round(precision_rf_tuned, 4))
print("Recall   :", round(recall_rf_tuned, 4))
print("F1 Score :", round(f1_rf_tuned, 4))
print("ROC-AUC  :", round(roc_auc_rf_tuned, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

Random Forest Results
---------------------
Accuracy : 0.7918
Precision: 0.6311
Recall   : 0.5214
F1 Score : 0.571
ROC-AUC  : 0.8396

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.52      0.57       374

    accuracy                           0.79      1407
   macro avg       0.73      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407



## **SMOTEENN IT FURTHER**

In [72]:
sm = SMOTEENN()

X_train_resampled, y_train_resampled = sm.fit_resample(
    X_train_processed,
    y_train
)

xr_train1,xr_test1,yr_train1,yr_test1=train_test_split(X_train_resampled, y_train_resampled,test_size=0.2)

model_rf_smote=RandomForestClassifier(bootstrap= True, n_estimators=100, random_state = 100, criterion='gini', max_depth=10, min_samples_leaf=1, max_features= 'sqrt', min_samples_split= 2)

# 4. Train the model
model_rf_smote.fit(X_train_resampled, y_train_resampled)

# 5. Evaluate on the untouched test set
y_pred = model_rf_smote.predict(X_test_processed)
y_prob = model_rf_smote.predict_proba(X_test_processed)[:, 1]

In [73]:
print("Random Forest Results (SMOOTHEN)")
print("---------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Random Forest Results (SMOOTHEN)
---------------------
Accuracy : 0.7171286425017769
Precision: 0.4811320754716981
Recall   : 0.8181818181818182
F1 Score : 0.6059405940594059
ROC-AUC  : 0.8368194501245012

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.68      0.78      1033
           1       0.48      0.82      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.69      1407
weighted avg       0.80      0.72      0.73      1407



# ***XGBOOST***

In [74]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_processed, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [75]:
y_pred_xgb = xgb_model.predict(X_test_processed)
y_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

In [76]:
print("XGBoost Results")
print("---------------------")

print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1 Score :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Results
---------------------
Accuracy : 0.7619047619047619
Precision: 0.5565217391304348
Recall   : 0.5133689839572193
F1 Score : 0.5340751043115438
ROC-AUC  : 0.8181145202954895

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.85      0.84      1033
           1       0.56      0.51      0.53       374

    accuracy                           0.76      1407
   macro avg       0.69      0.68      0.69      1407
weighted avg       0.76      0.76      0.76      1407



In [77]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train_processed, y_train)

best_xgb_model = grid_search.best_estimator_

print(grid_search.best_params_)
print(grid_search.best_score_)

{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
0.6126157544869766


In [78]:
best_xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    learning_rate=0.05,
    n_estimators=200,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8
)

In [79]:
best_xgb_model.fit(X_train_processed, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [80]:
y_pred = best_xgb_model.predict(X_test_processed)
y_prob = best_xgb_model.predict_proba(X_test_processed)[:,1]

print("Tuned XGBoost Results")
print("---------------------")

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print(classification_report(y_test, y_pred))

Tuned XGBoost Results
---------------------
Accuracy : 0.7953091684434968
Precision: 0.634375
Recall   : 0.5427807486631016
F1 Score : 0.5850144092219021
ROC-AUC  : 0.8478938349959362
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.54      0.59       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407



In [81]:
thresholds = np.arange(0.2,0.81,0.01)

best_f1 = 0
best_threshold = 0

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    score = f1_score(y_test, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

final_preds = (y_prob >= best_threshold).astype(int)

In [82]:
print("\nThreshold Optimized XGBoost Results")
print("-----------------------------------")
print("Accuracy :", accuracy_score(y_test, final_preds))
print("Precision:", precision_score(y_test, final_preds))
print("Recall   :", recall_score(y_test, final_preds))
print("F1 Score :", f1_score(y_test, final_preds))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, final_preds))


Threshold Optimized XGBoost Results
-----------------------------------
Accuracy : 0.7846481876332623
Precision: 0.5773420479302832
Recall   : 0.7085561497326203
F1 Score : 0.6362545018007203
ROC-AUC  : 0.8478938349959362

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.81      0.85      1033
           1       0.58      0.71      0.64       374

    accuracy                           0.78      1407
   macro avg       0.73      0.76      0.74      1407
weighted avg       0.80      0.78      0.79      1407



## **COMPAIRING THE MODELS**

In [83]:
results = [
    {
        "Model": "Logistic Regression",
        "Variant": "Baseline",
        "Accuracy": 0.7967,
        "Precision": 0.6279,
        "Recall": 0.5775,
        "F1": 0.6017,
        "ROC-AUC": 0.8447
    },
    {
        "Model": "Logistic Regression",
        "Variant": "Hyperparameter Tuned",
        "Accuracy": 0.7946,
        "Precision": 0.6246,
        "Recall": 0.5695,
        "F1": 0.5958,
        "ROC-AUC": 0.8443
    },
    {
        "Model": "Logistic Regression",
        "Variant": "SMOTE",
        "Accuracy": 0.7306,
        "Precision": 0.4958,
        "Recall": 0.7941,
        "F1": 0.6105,
        "ROC-AUC": 0.8422
    },
    {
        "Model": "Logistic Regression",
        "Variant": "Threshold Optimized",
        "Accuracy": 0.8003,
        "Precision": 0.6177,
        "Recall": 0.6524,
        "F1": 0.6346,
        "ROC-AUC": 0.8443
    },

    {
        "Model": "Decision Tree",
        "Variant": "Baseline",
        "Accuracy": 0.7875,
        "Precision": 0.5984,
        "Recall": 0.6096,
        "F1": 0.6040,
        "ROC-AUC": 0.8348
    },
    {
        "Model": "Decision Tree",
        "Variant": "Hyperparameter Tuned",
        "Accuracy": 0.7868,
        "Precision": 0.6000,
        "Recall": 0.5936,
        "F1": 0.5968,
        "ROC-AUC": 0.8220
    },
    {
        "Model": "Decision Tree",
        "Variant": "Hyperparameter + SMOTEENN",
        "Accuracy": 0.7029,
        "Precision": 0.4666,
        "Recall": 0.8209,
        "F1": 0.5950,
        "ROC-AUC": 0.8003
    },
    {
        "Model": "Random Forest",
        "Variant": "Baseline",
        "Accuracy": 0.7918,
        "Precision": 0.6494,
        "Recall": 0.4706,
        "F1": 0.5457,
        "ROC-AUC": 0.8452
    },
    {
        "Model": "Random Forest",
        "Variant": "Hyperparameter + SMOTEENN",
        "Accuracy": 0.7214,
        "Precision": 0.4857,
        "Recall": 0.8155,
        "F1": 0.6088,
        "ROC-AUC": 0.8382
    },
    {
        "Model": "XGBoost",
        "Variant": "Baseline",
        "Accuracy": 0.7953091684434968,
        "Precision": 0.634375,
        "Recall": 0.5427807486631016,
        "F1": 0.5850144092219021,
        "ROC-AUC": 0.8478938349959362
    },
    {   
        "Model": "XGBoost",
        "Variant": "Threshold Optimized",
        "Accuracy": 0.7846481876332623,
        "Precision": 0.5773420479302832,
        "Recall": 0.7085561497326203,
        "F1": 0.6362545018007203,
        "ROC-AUC": 0.8478938349959362
    }
]

results_df = pd.DataFrame(results)
results_df

,Model,Variant,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,Baseline,0.796700,0.627900,0.577500,0.601700,0.844700
1,Logistic Regression,Hyperparameter Tuned,0.794600,0.624600,0.569500,0.595800,0.844300
2,Logistic Regression,SMOTE,0.730600,0.495800,0.794100,0.610500,0.842200
3,Logistic Regression,Threshold Optimized,0.800300,0.617700,0.652400,0.634600,0.844300
4,Decision Tree,Baseline,0.787500,0.598400,0.609600,0.604000,0.834800
5,Decision Tree,Hyperparameter Tuned,0.786800,0.600000,0.593600,0.596800,0.822000
6,Decision Tree,Hyperparameter + SMOTEENN,0.702900,0.466600,0.820900,0.595000,0.800300
7,Random Forest,Baseline,0.791800,0.649400,0.470600,0.545700,0.845200
8,Random Forest,Hyperparameter + SMOTEENN,0.721400,0.485700,0.815500,0.608800,0.838200
9,XGBoost,Baseline,0.795309,0.634375,0.542781,0.585014,0.847894


#### **Model Selection: Multiple machine learning models, including Logistic Regression, Decision Tree, Random Forest, and XGBoost, were evaluated using baseline, hyperparameter tuning, resampling techniques (SMOTE/SMOTEENN), and threshold optimization. Among all the evaluated models, Threshold-Optimized Logistic Regression was selected as the final model as it provided the best balance between predictive performance and interpretability. It achieved an accuracy of 80.03%, precision of 61.77%, recall of 65.24%, F1-score of 63.46%, and a ROC-AUC of 84.43%. Although Threshold-Optimized XGBoost produced a marginally higher F1-score (63.63%) and ROC-AUC (84.79%), the improvement was minimal. Logistic Regression offered comparable performance while remaining computationally efficient, easier to interpret, and well-suited for generating explainable predictions using SHAP. Therefore, Threshold-Optimized Logistic Regression was selected as the final model for the customer churn prediction system.**

## **Production Pipeline for Flask Deployment**

In [90]:
production_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", best_log_model)
    ]
)

In [91]:
production_pipeline.fit(X_train, y_train)

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [ ]:
threshold = 0.45000000000000023

y_prob_pipeline = production_pipeline.predict_proba(X_test)[:, 1]
y_pred_pipeline = (y_prob_pipeline >= threshold).astype(int)

In [93]:
accuracy = accuracy_score(y_test, y_pred_pipeline)
precision = precision_score(y_test, y_pred_pipeline)
recall = recall_score(y_test, y_pred_pipeline)
f1 = f1_score(y_test, y_pred_pipeline)
roc_auc = roc_auc_score(y_test, y_prob_pipeline)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred_pipeline))

# print("\nConfusion Matrix")
# print(confusion_matrix(y_test, y_pred_pipeline))

Accuracy : 0.8003
Precision: 0.6177
Recall   : 0.6524
F1 Score : 0.6346
ROC-AUC  : 0.8443

Classification Report
              precision    recall  f1-score   support

           0       0.87      0.85      0.86      1033
           1       0.62      0.65      0.63       374

    accuracy                           0.80      1407
   macro avg       0.74      0.75      0.75      1407
weighted avg       0.80      0.80      0.80      1407



In [95]:
os.makedirs("../model", exist_ok=True)

joblib.dump(
    production_pipeline,
    "../model/churn_pipeline.pkl"
)

print("✅ Production pipeline saved successfully!")

✅ Production pipeline saved successfully!


In [97]:
joblib.dump(0.45000000000000023, "../model/threshold.pkl")

['../model/threshold.pkl']